<a href="https://colab.research.google.com/github/crystalloide/Kafka/blob/main/kafka_4_1_installation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Installation d'Apache Kafka 4.0 en Mode KRaft Standalone

Ce notebook installe et configure Apache Kafka 4.0 sur Ubuntu 24.04 en utilisant le mode KRaft (Kafka Raft Metadata), qui remplace Zookeeper.

**Prérequis:**
- Ubuntu 24.04 avec accès sudo
- Minimum 2 GB de RAM
- Java 21 (sera installé)

## 1. Mise à jour du système et installation de Java

In [ ]:
# Mettre à jour la liste des paquets
!sudo apt update

# Installer OpenJDK 21
!sudo apt install -y openjdk-21-jdk

# Vérifier l'installation de Java
!java -version

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:7 https://cli.github.com/packages stable InRelease
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,592 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packag

## 2. Créer un utilisateur dédié et les répertoires Kafka

In [ ]:
# Créer l'utilisateur système 'kafka' sans accès de connexion
!sudo useradd -r -m -U -d /opt/kafka -s /bin/false kafka 2>/dev/null || echo "L'utilisateur 'kafka' existe déjà"

# Créer le répertoire principal de Kafka
!sudo mkdir -p /opt/kafka

# Attribuer la propriété du répertoire à l'utilisateur kafka
!sudo chown -R kafka:kafka /opt/kafka

# Vérifier les permissions
!ls -ld /opt/kafka

## 3. Télécharger et extraire Kafka 4.1.1

In [ ]:
import os
import subprocess

# Se placer dans le répertoire temporaire
os.chdir('/tmp')

# Télécharger Kafka 4.0.0
print("Téléchargement de Kafka 4.1.1...")
!wget -q https://downloads.apache.org/kafka/4.1.1/kafka_2.13-4.1.1.tgz
print("✓ Téléchargement terminé")

# Extraire l'archive
print("\nExtraction de l'archive...")
!sudo tar -xzf kafka_2.13-4.1.1.tgz -C /opt/kafka --strip-components=1
print("✓ Extraction terminée")

# Mettre à jour les permissions
!sudo chown -R kafka:kafka /opt/kafka

# Vérifier le contenu
print("\nContenu de /opt/kafka:")
!sudo ls -l /opt/kafka

## 4. Configuration de Kafka en mode KRaft

In [ ]:
# Lire le fichier de configuration actuel
print("Configuration actuelle de Kafka...\n")

# Vérifier les paramètres KRaft clés
!sudo grep -E "process.roles|node.id|controller.quorum" /opt/kafka/config/server.properties | head -5

In [ ]:
# Créer une sauvegarde du fichier de configuration original
!sudo cp /opt/kafka/config/server.properties /opt/kafka/config/server.properties.bak

# Configuration KRaft pour mode standalone
kraft_config = """# Mode KRaft - Broker et Controller sur le même nœud
process.roles=broker,controller
node.id=1

# Quorum Controller
controller.quorum.bootstrap.servers=localhost:9093
controller.quorum.voters=1@localhost:9093

# Listeners
listeners=PLAINTEXT://:9092,CONTROLLER://:9093
advertised.listeners=PLAINTEXT://localhost:9092,CONTROLLER://localhost:9093
controller.listener.names=CONTROLLER

# Protocoles de sécurité
listener.security.protocol.map=CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT,SSL:SSL,SASL_PLAINTEXT:SASL_PLAINTEXT,SASL_SSL:SASL_SSL

# Répertoire de données
log.dirs=/opt/kafka/data

# Autres paramètres essentiels
offsets.topic.replication.factor=1
transaction.state.log.replication.factor=1
transaction.state.log.min.isr=1
"""

# Écrire la configuration dans un fichier temporaire
with open('/tmp/kraft_config.txt', 'w') as f:
    f.write(kraft_config)

# Ajouter la configuration au fichier server.properties
!sudo bash -c 'cat /tmp/kraft_config.txt >> /opt/kafka/config/server.properties'

print("✓ Configuration KRaft ajoutée")
print("\nVérification de la configuration:")
!sudo grep -E "process.roles|node.id|controller.quorum.voters|log.dirs" /opt/kafka/config/server.properties | tail -4

## 5. Générer un Cluster ID et formater le stockage

In [ ]:
# Générer un Cluster ID unique
print("Génération du Cluster ID...\n")
result = !sudo /opt/kafka/bin/kafka-storage.sh random-uuid
cluster_id = str(result[0]).strip()
print(f"Cluster ID généré: {cluster_id}\n")

# Stocker le Cluster ID pour référence
with open('/tmp/cluster_id.txt', 'w') as f:
    f.write(cluster_id)

print("Cluster ID sauvegardé dans /tmp/cluster_id.txt")

In [ ]:
# Lire le Cluster ID sauvegardé
with open('/tmp/cluster_id.txt', 'r') as f:
    cluster_id = f.read().strip()

print(f"Utilisation du Cluster ID: {cluster_id}\n")

# Formater le répertoire de stockage Kafka
print("Formatage du répertoire de stockage...\n")
!sudo /opt/kafka/bin/kafka-storage.sh format -t {cluster_id} -c /opt/kafka/config/server.properties

print("\n✓ Stockage formaté avec succès")

## 6. Créer un service systemd pour Kafka

In [ ]:
# Contenu du fichier de service systemd
service_content = """[Unit]
Description=Apache Kafka Server
After=network.target

[Service]
Type=simple
User=kafka
ExecStart=/opt/kafka/bin/kafka-server-start.sh /opt/kafka/config/server.properties
ExecStop=/opt/kafka/bin/kafka-server-stop.sh
Restart=on-abnormal

[Install]
WantedBy=multi-user.target
"""

# Écrire le fichier de service
with open('/tmp/kafka.service', 'w') as f:
    f.write(service_content)

# Copier le fichier de service vers le répertoire systemd
!sudo cp /tmp/kafka.service /etc/systemd/system/kafka.service

# Vérifier le contenu du service
print("Contenu du service Kafka:")
!sudo cat /etc/systemd/system/kafka.service

## 7. Créer le répertoire de données et démarrer Kafka

In [ ]:
# Créer le répertoire de données Kafka
print("Création du répertoire de données...")
!sudo mkdir -p /opt/kafka/data
!sudo chown -R kafka:kafka /opt/kafka/data

# Vérifier les permissions
!ls -ld /opt/kafka/data

print("\n✓ Répertoire de données créé")

In [ ]:
# Recharger systemd pour reconnaître le nouveau service
print("Rechargement de systemd...")
!sudo systemctl daemon-reload

# Activer Kafka au démarrage du système
print("Activation de Kafka au démarrage...")
!sudo systemctl enable kafka

# Démarrer le service Kafka
print("Démarrage du service Kafka...")
!sudo systemctl restart kafka

print("\n✓ Kafka démarré")

In [ ]:
# Vérifier le statut de Kafka
print("Statut du service Kafka:\n")
!sudo systemctl status kafka

## 8. Tests de Kafka

In [ ]:
import time

# Attendre que Kafka soit complètement prêt
print("Attente du démarrage complet de Kafka (10 secondes)...")
time.sleep(10)

# Créer un topic de test
print("\nCréation d'un topic de test 'test-topic'...")
!sudo /opt/kafka/bin/kafka-topics.sh --create --topic test-topic --bootstrap-server localhost:9092 --partitions 1 --replication-factor 1

print("\n✓ Topic créé")

In [ ]:
# Lister les topics
print("Topics disponibles:\n")
!sudo /opt/kafka/bin/kafka-topics.sh --list --bootstrap-server localhost:9092

In [ ]:
# Afficher les détails du topic
print("Détails du topic 'test-topic':\n")
!sudo /opt/kafka/bin/kafka-topics.sh --describe --topic test-topic --bootstrap-server localhost:9092

## 9. Résumé et Informations Utiles

In [ ]:
print("="*70)
print("INSTALLATION D'APACHE KAFKA 4.0 TERMINÉE")
print("="*70)

print("\n📍 CHEMINS IMPORTANTS:")
print("   - Installation: /opt/kafka")
print("   - Configuration: /opt/kafka/config/server.properties")
print("   - Données: /opt/kafka/data")
print("   - Service systemd: /etc/systemd/system/kafka.service")

print("\n⚙️  MODE DE FONCTIONNEMENT:")
print("   - Mode KRaft (Kafka Raft Metadata)")
print("   - Broker & Controller sur le même nœud")
print("   - Node ID: 1")

print("\n🔌 PORTS:")
print("   - Clients (PLAINTEXT): 9092")
print("   - Controller (interne): 9093")

print("\n📋 COMMANDES UTILES:")
print("   - Démarrer: sudo systemctl start kafka")
print("   - Arrêter: sudo systemctl stop kafka")
print("   - Statut: sudo systemctl status kafka")
print("   - Logs: sudo journalctl -u kafka -f")

print("\n📝 CRÉER UN TOPIC:")
print("   sudo /opt/kafka/bin/kafka-topics.sh --create \\")
print("     --topic mon-topic \\")
print("     --bootstrap-server localhost:9092 \\")
print("     --partitions 3 --replication-factor 1")

print("\n📤 PRODUIRE DES MESSAGES:")
print("   sudo /opt/kafka/bin/kafka-console-producer.sh \\")
print("     --topic mon-topic \\")
print("     --bootstrap-server localhost:9092")

print("\n📥 CONSOMMER DES MESSAGES:")
print("   sudo /opt/kafka/bin/kafka-console-consumer.sh \\")
print("     --topic mon-topic \\")
print("     --from-beginning \\")
print("     --bootstrap-server localhost:9092")

print("\n✅ Kafka 4.0 est prêt pour développement et tests!")
print("="*70)